In [0]:
%pip install --quiet beautifulsoup4 curl_cffi httpx lxml playwright
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# =============================================================================
# Imports
# =============================================================================
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
import concurrent.futures
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
# =============================================================================
# Configuração — fonte lida via parâmetro do Job (widget), não fixa no código
# =============================================================================

dbutils.widgets.text("fonte", "todas")
NOME_FONTE = dbutils.widgets.get("fonte")

# Registro central de fontes "site inteiro". modo="http" usa curl_cffi/httpx;
# modo="navegador" usa Playwright (Chrome compartilhado do Selenium) — só
# necessário pra fontes com proteção anti-bot, como o PPI.
CONFIGS_FONTES = {
    "estadao": {
        "site_url": "https://www.estadao.com.br/",
        "modo": "http",
    },
    "o_globo": {
        "site_url": "https://oglobo.globo.com/",
        "modo": "http",
    },
    "valor_infra": {
        "site_url": "https://valor.globo.com/brasil/infraestrutura/",
        "modo": "http",
    },
    "moodys": {
        "site_url": "https://moodyslocal.com.br/",
        "modo": "http",
    },
    "agencia_eixos": {
        "site_url": "https://eixos.com.br/",
        "modo": "http",
    },
    "agencia_infra": {
        "site_url": "https://agenciainfra.com/blog/",
        "modo": "http",
    },
    "ppi": {
        "site_url": "https://ppi.gov.br/noticias/",
        "modo": "navegador",  # site atrás de proteção anti-bot — precisa renderizar
    },
}

HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)
print(f"[setup] Salvando artefatos em: {PASTA_DESTINO}")

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:126.0) Gecko/20100101 Firefox/126.0",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

HTTP_TIMEOUT = 30

# Chrome já provisionado no Volume (mesma infra usada pelo Selenium do time) —
# evita depender de "playwright install", que falha por falta de permissão root.
CHROME_BIN = "/tmp/chrome/chrome-linux64/chrome"


[setup] Salvando artefatos em: /Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/2026-08-01


In [0]:
# =============================================================================
# Helpers
# =============================================================================

def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def domain_from_url(url: str) -> str:
    try:
        netloc = urllib.parse.urlparse(url).netloc.lower()
        return netloc[4:] if netloc.startswith("www.") else netloc
    except Exception:
        return "desconhecido"


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": random.choice(USER_AGENTS),
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


In [0]:
# =============================================================================
# Etapa 1 — Baixar a página (modo "http" ou "navegador")
# =============================================================================

def baixar_html_http(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=url)

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def _baixar_html_playwright_worker(url: str, resultado: dict) -> None:
    from playwright.sync_api import sync_playwright

    with sync_playwright() as p:
        browser = p.chromium.launch(executable_path=CHROME_BIN, headless=True)
        try:
            page = browser.new_page(user_agent=random.choice(USER_AGENTS))
            page.goto(url, timeout=30000, wait_until="networkidle")
            resultado["html"] = page.content()
        finally:
            browser.close()


def baixar_html_navegador(url: str, tentativas: int = 2) -> Optional[str]:
    if not os.path.exists(CHROME_BIN):
        print(f"  [playwright] Chrome não encontrado em {CHROME_BIN!r}. "
              f"O init script 'init_selenium2_desafio.sh' provavelmente não "
              f"está anexado a este cluster.")
        return None

    for tentativa in range(1, tentativas + 1):
        try:
            resultado = {}
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                executor.submit(_baixar_html_playwright_worker, url, resultado).result(timeout=45)
            if resultado.get("html") and len(resultado["html"]) > 500:
                return resultado["html"]
        except Exception as e:
            print(f"  [playwright tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def baixar_html(url: str, modo: str) -> Optional[str]:
    if modo == "navegador":
        return baixar_html_navegador(url)
    return baixar_html_http(url)


In [0]:
# =============================================================================
# Etapa 2 — Extrair todos os links da página
# =============================================================================

def extrair_links(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    links = []
    vistos = set()

    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()

        if (not href or href.startswith("#") or href.startswith("javascript:")
                or href.startswith("mailto:") or href.startswith("tel:")):
            continue

        url_absoluta = urllib.parse.urljoin(url_base, href)

        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)

        texto_ancora = tag_a.get_text(" ", strip=True)
        links.append({"texto_ancora": texto_ancora, "url": url_absoluta})

    return links


In [0]:
# =============================================================================
# Etapa 3 — Limpar o HTML e extrair só o texto útil
# =============================================================================

TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form",
             "nav", "footer", "header", "aside", "button"]


def extrair_texto(html: str) -> str:
    if not html:
        return ""

    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception as e:
        print(f"    -> lxml falhou ao parsear ({e}); usando html.parser como fallback.")
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    articles = soup.find_all("article")

    if len(articles) == 1:
        texto_article = articles[0].get_text("\n", strip=True)
        texto = texto_article if len(texto_article) > 500 else soup.get_text("\n", strip=True)
    else:
        texto = soup.get_text("\n", strip=True)

    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()


def extrair_titulo(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
        if soup.title and soup.title.string:
            return soup.title.string.strip()
    except Exception:
        pass
    return ""


In [0]:
# =============================================================================
# Etapa 4 — Salvar no Volume
# =============================================================================

def salvar_artefatos(pasta, source, titulo, texto, links, metadados):
    slug_source = slugify(source, max_len=40) or "fonte"
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_links = os.path.join(pasta, f"{nome_base}_links.json")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")

    with open(caminho_links, "w", encoding="utf-8") as f:
        json.dump(links, f, ensure_ascii=False, indent=2)

    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_links, caminho_json


In [0]:
# =============================================================================
# Etapa 5 — Pipeline principal (uma fonte)
# =============================================================================

def processar_site(url: str, modo: str, pasta_destino: str) -> Optional[dict]:
    print(f"\n=== Site: {url!r} (modo={modo}) ===")

    html = baixar_html(url, modo)
    if not html:
        print("  -> download do HTML falhou; abortando.")
        return None
    print(f"  HTML baixado ({len(html)} chars).")

    links = extrair_links(html, url_base=url)
    print(f"  {len(links)} links únicos encontrados.")

    texto = extrair_texto(html)
    print(f"  Texto extraído ({len(texto)} chars).")

    source = domain_from_url(url)
    titulo = extrair_titulo(html) or source
    metadados = {
        "source_id": "site_page",
        "title": titulo,
        "description": f"Página coletada diretamente do site: {url}",
        "url": url,
        "date": HOJE,
        "qtd_links": len(links),
        "qtd_chars_texto": len(texto),
    }

    caminho_txt, caminho_links, caminho_json = salvar_artefatos(
        pasta=pasta_destino, source=source, titulo=titulo,
        texto=texto, links=links, metadados=metadados,
    )
    print(f"  -> salvo em {caminho_txt}")

    return {
        "url": url, "titulo": titulo, "source": source,
        "qtd_links": len(links), "qtd_chars_texto": len(texto),
        "caminho_txt": caminho_txt, "caminho_links": caminho_links,
        "caminho_json": caminho_json,
    }


In [0]:
# =============================================================================
# Execução — roda todas as fontes de CONFIGS_FONTES em sequência
# (ou só uma, se "fonte" for passado com um nome específico em vez de "todas")
# =============================================================================

if NOME_FONTE == "todas":
    fontes_a_rodar = CONFIGS_FONTES
else:
    if NOME_FONTE not in CONFIGS_FONTES:
        raise ValueError(f"Fonte {NOME_FONTE!r} não configurada. Opções: {list(CONFIGS_FONTES)}")
    fontes_a_rodar = {NOME_FONTE: CONFIGS_FONTES[NOME_FONTE]}

resumo_geral = {}

for nome_fonte, config in fontes_a_rodar.items():
    try:
        resultado = processar_site(config["site_url"], config["modo"], PASTA_DESTINO)
        resumo_geral[nome_fonte] = "OK" if resultado else "FALHOU"
    except Exception as e:
        print(f"[ERRO GERAL] fonte {nome_fonte!r} falhou por completo: {e}")
        resumo_geral[nome_fonte] = f"ERRO: {e}"

print(f"\n\n{'='*70}")
print("=== RESUMO FINAL ===")
for nome_fonte, resultado in resumo_geral.items():
    print(f"  {nome_fonte}: {resultado}")
print(f"{'='*70}")



=== Site: 'https://www.estadao.com.br/' (modo=http) ===
  HTML baixado (1518591 chars).
  202 links únicos encontrados.
  Texto extraído (14595 chars).
  -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/2026-08-01/estadao-com-br_estadao-as-ultimas-noticias-do-brasil-e-do-mundo-estadao_b5d8715f.txt

=== Site: 'https://oglobo.globo.com/' (modo=http) ===
  HTML baixado (916236 chars).
  250 links únicos encontrados.
  Texto extraído (9590 chars).
  -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/2026-08-01/oglobo-globo-com_o-globo-confira-as-principais-noticias-do-brasil-e-do-mundo_9fad2674.txt

=== Site: 'https://valor.globo.com/brasil/infraestrutura/' (modo=http) ===
  HTML baixado (745474 chars).
  187 links únicos encontrados.
  Texto extraído (2707 chars).
  -> salvo em /Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/2026-08-01/valor-globo-com_infraestrutura_89adbda5.txt

=== Site: 'https://moodysl